# Geo Plotting

`kandy-geo` adds extensions which allow to plot a `GeoDataFrame` — a [DataFrame](https://github.com/Kotlin/dataframe)-based structure for geospatial datasets. [All the plotting principles and features](https://kotlin.github.io/kandy/quick-start-guide.html) here are the same as Kandy, the only difference is that you don't have to perform positional mapping — instead geometries will be automatically mapped to Kandy layers.

## Kandy-geo and GeoDataFrame Usage

In [1]:
%useLatestDescriptors
// Add both Kandy-Geo and DataFrame-Geo,
// as well as base Kandy and Kotlin DataFrame
%use kandy-geo

## Geometries

Geo plotting is essentially the visualization of geographic geometries on a map or coordinate system. [GeoJSON](https://en.wikipedia.org/wiki/GeoJSON) is the most widely used standard for representing geospatial data. It defines a set of geometry types that are simple yet powerful for modeling geographic features:

- **`Point`**: Represents a specific location as a single coordinate.
- **`MultiPoint`**: A collection of multiple `Point` geometries.
- **`LineString`**: A sequence of connected points, forming a path or linear feature.
- **`MultiLineString`**: A collection of multiple `LineString` geometries.
- **`Polygon`**: A closed shape with an outer boundary and optional inner holes.
- **`MultiPolygon`**: A collection of multiple `Polygon` geometries.
- **`GeometryCollection`**: A container for any combination of the above geometries.

 [JTS](https://github.com/locationtech/jts) (Java Topology Suite) is a library that works seamlessly with these geometries, adding a variety of operations. It allows you to perform tasks like combining geometries, finding intersections, creating buffers, or simplifying shapes.

All classes for the aforementioned geometries are provided in JTS and inherit from the base class `Geometry`. **`GeoDataFrame`** is a wrapper around a standard `DataFrame` with a `geometry` column of type `Geometry`, enabling convenient handling of geospatial datasets.



## Reading GeoDataFrame

Currently, the GeoDataFrame supports two of the most popular formats: Shapefile and GeoJSON. These formats can be read into a `GeoDataFrame` using the corresponding `GeoDataFrame.read..()` functions. Each of these functions returns a `GeoDataFrame`.

### GeoJSON

[GeoJSON](https://en.wikipedia.org/wiki/GeoJSON) is a widely used format for encoding geographic data structures. It represents spatial features such as points, lines, and polygons, along with their properties, using JSON. Here's an example of GeoJSON:

```
{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [125.6, 10.1]
  },
  "properties": {
    "name": "Dinagat Islands"
  }
}
```


Let's load a GeoJSON file that contains polygons representing the boundaries of US states:

In [2]:
val usaStates = GeoDataFrame.readGeoJson("https://raw.githubusercontent.com/AndreiKingsley/datasets/refs/heads/main/USA.json")

We can directly access the underlying DataFrame to take a closer look at its contents:

In [3]:
usaStates.df

name,geometry
Alabama,"POLYGON ((-87.359296 35.00118, -85.60..."
Alaska,MULTIPOLYGON (((-131.602021 55.117982...
Arizona,"POLYGON ((-109.042503 37.000263, -109..."
Arkansas,"POLYGON ((-94.473842 36.501861, -90.1..."
California,"POLYGON ((-123.233256 42.006186, -122..."
Colorado,"POLYGON ((-107.919731 41.003906, -105..."
Connecticut,"POLYGON ((-73.053528 42.039048, -71.7..."
Delaware,"POLYGON ((-75.414089 39.804456, -75.5..."
District of Columbia,"POLYGON ((-77.035264 38.993869, -76.9..."
Florida,"POLYGON ((-85.497137 30.997536, -85.0..."


This DataFrame **is required** to have a geometry column of type `org.locationtech.jts.geom.Geometry`:



In [4]:
usaStates.df.geometry.type()

org.locationtech.jts.geom.Geometry

We can also check the exact types of these geometries:

In [5]:
usaStates.df.geometry.map { it::class }.distinct().toList()

[class org.locationtech.jts.geom.Polygon, class org.locationtech.jts.geom.MultiPolygon]

As expected, these are `Polygon` and `MultiPolygon`.

The `GeoDataFrame` also contains a `.crs` field for the coordinate reference system (CRS). In GeoJSON, this field is not explicitly defined* and is read as `null`. If this field is not explicitly set in the GeoDataFrame, it is assumed by default to use [WGS84]([https://gisgeography.com/wgs84-world-geodetic-system/](https://gisgeography.com/wgs84-world-geodetic-system/)) — the standard CRS for working with geospatial data.

\* *According to the [GeoJSON specification](https://datatracker.ietf.org/doc/html/rfc7946#section-4), all coordinates are defined in WGS84. In the future, we may remove the nullability of the `crs` field, and WGS84 will be explicitly set as the CRS when reading GeoJSON files.*



In [6]:
usaStates.crs

null

### Shapefile

[Shapefile]() is a popular geospatial vector data format developed by ESRI. It stores geometric features such as points, lines, and polygons, along with their attributes, across multiple files. A Shapefile requires at least three parts: `.shp` (geometry), `.shx` (spatial index), and `.dbf` (attributes), and it typically uses a defined coordinate reference system.

To load a Shapefile, you need to specify the path to the file with the `.shp` extension. The other required files must be in the same directory and share the same base name.

Let's load a Shapefile with the most populated cities in the world:


In [7]:
val worldCities = GeoDataFrame.readShapefile("https://github.com/AndreiKingsley/datasets/raw/refs/heads/main/ne_10m_populated_places_simple/ne_10m_populated_places_simple.shp")

Take a look inside the DataFrame:

In [8]:
worldCities.df

scalerank,natscale,labelrank,featurecla,name,namepar,namealt,nameascii,adm0cap,capalt,capin,worldcity,megacity,sov0name,sov_a3,adm0name,adm0_a3,adm1name,iso_a2,note,latitude,longitude,pop_max,pop_min,pop_other,rank_max,rank_min,meganame,ls_name,min_zoom,ne_id,geometry
10,1,8,Admin-1 capital,Colonia del Sacramento,,,Colonia del Sacramento,0,0,,0,0,Uruguay,URY,Uruguay,URY,Colonia,UY,,"-34,479999","-57,840003",21714,21714,0,7,7,,,"9,000000",1159112629,POINT (-57.836116004496425 -34.469787...
10,1,8,Admin-1 capital,Trinidad,,,Trinidad,0,0,,0,0,Uruguay,URY,Uruguay,URY,Flores,UY,,"-33,543999","-56,900997",21093,21093,0,7,7,,,"9,000000",1159112647,POINT (-56.9009966 -33.5439989)
10,1,8,Admin-1 capital,Fray Bentos,,,Fray Bentos,0,0,,0,0,Uruguay,URY,Uruguay,URY,Río Negro,UY,,"-33,138999","-58,303998",23279,23279,0,7,7,,,"9,000000",1159112663,POINT (-58.3039975 -33.138999)
10,1,8,Admin-1 capital,Canelones,,,Canelones,0,0,,0,0,Uruguay,URY,Uruguay,URY,Canelones,UY,,"-34,538004","-56,284002",19698,19698,0,6,6,,,"9,000000",1159112679,POINT (-56.2840015 -34.538004)
10,1,8,Admin-1 capital,Florida,,,Florida,0,0,,0,0,Uruguay,URY,Uruguay,URY,Florida,UY,,"-34,099002","-56,214998",32234,32234,0,7,7,,,"7,000000",1159112703,POINT (-56.2149984 -34.099002)
10,1,8,Admin-1 capital,Bassar,,,Bassar,0,0,,0,0,Togo,TGO,Togo,TGO,Kara,TG,,"9,261000","0,789004",61845,61845,0,8,8,,,"9,000000",1159112719,POINT (0.7890036 9.2610001)
10,1,8,Admin-1 capital,Sotouboua,,,Sotouboua,0,0,,0,0,Togo,TGO,Togo,TGO,Centre,TG,,"8,557002","0,984997",21054,21054,0,7,7,,,"7,000000",1159112735,POINT (0.9849965 8.5570021)
10,1,7,Admin-1 capital,Medenine,,,Medenine,0,0,,0,0,Tunisia,TUN,Tunisia,TUN,MUdenine,TN,,"33,399999","10,416700",61705,61705,0,8,8,,,"6,100000",1159112749,POINT (10.4166996 33.399999)
10,1,7,Admin-1 capital,Kebili,,,Kebili,0,0,,0,0,Tunisia,TUN,Tunisia,TUN,Kebili,TN,,"33,689997","8,971003",19875,19875,0,6,6,,,"9,000000",1159112765,POINT (8.9710025 33.689997)
10,1,7,Admin-1 capital,Tataouine,,,Tataouine,0,0,,0,0,Tunisia,TUN,Tunisia,TUN,Tataouine,TN,,"33,000003","10,466704",62577,62577,0,8,8,,,"9,000000",1159112779,POINT (10.4667036 33.0000032)


This `GeoDataFrame` contains only `Point` geometries:


In [9]:
worldCities.df.geometry.type()

org.locationtech.jts.geom.Point

And has explicitly specified CRS:

In [10]:
worldCities.crs

GEOGCS["GCS_WGS_1984", 
  DATUM["D_WGS_1984", 
    SPHEROID["WGS_1984", 6378137.0, 298.257223563]], 
  PRIMEM["Greenwich", 0.0], 
  UNIT["degree", 0.017453292519943295], 
  AXIS["Longitude", EAST], 
  AXIS["Latitude", NORTH]]

## Plot

Geoplotting in Kandy is not significantly different from usual plotting. The main distinction is that you need to provide the aforementioned geometries instead of specifying positional mappings.

To facilitate this, Kandy-geo introduces *geo layers*, which, unlike regular layers, accept geometries. These can be provided as DataFrame columns, `Iterable`, or single instances. If a layer is built in the context of a `GeoDataFrame` dataset, it is not necessary to explicitly specify the geometry, as the `geometry` column will be used by default.



### geoPolygon

The `geoPolygon()` adds a layer of polygons constructed using `Polygon` and `MultiPolygon` geometries.


Let's plot US states from `usaStates`:

In [11]:
usaStates.plot {
    // `geoPolygon` uses polygons and multipolygons
    // from `geometry` column of `usaStates` inner DataFrame.
    geoPolygon()
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="xosiBm"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-133.102702,57.007526],[-132.932917,56.82131],[-132.620732,56.667956],[-132.653593,56.55294],[-132.817901,56.492694],[-133.042456,56.520078],[-133.201287,56.448878],[-133.420365,56.492694],[-133.66135,56.448878],[-133.710643,56.684386],[-133.688735,56.837741],[-133.869474,56.843218],[-133.907813,56.930849]]],[[[-134.115936,56.48174],[-134.25286,56.558417],[-134.400737,56.722725],[-134.417168,56.848695],[-134.296675,56.908941],[-134.170706,56.848695],[-134.143321,56.952757],[-133.748981,56.772017],[-133.710643,56.596755],[-133.847566,56.574848],[-133.935197,56.377678],[-133.836612,56.322908],[-133.957105,56.092877],[-134.110459,56.142169],[-134.132367,55.999769],[-134.230952,56.070969],[-134.291198,56.350293],[-134

The customization process for such a layer is no different from a regular one. The function optionally opens a block where you can configure all polygon aesthetic attributes as usual using mappings and settings. For example, you can color each state by mapping the `name` column to `fillColor` and customize the `borderLine` as shown below:


In [12]:
usaStates.plot {
    geoPolygon() {
        fillColor(name) { legend.type = LegendType.None } // hide legend
        borderLine {
            width = 0.1
            color = Color.BLACK
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JT94UE"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true,
"guide":"none"
}],
"layers":[{
"mapping":{
"fill":"name"
},
"stat":"identity",
"size":0.1,
"color":"#000000",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"name":["Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut","Delaware","District of Columbia","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa","Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan","Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire","New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota","Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia","Wisconsin","Wyoming","Puerto Rico"],
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-13

#### Mercator coordinates transformation

The Mercator projection is widely used for map visualizations because it preserves angles and shapes locally, making it ideal for navigation and geographical applications. It is particularly useful for rendering maps on flat surfaces, such as screens or paper. The Mercator projection is compatible with coordinates in the WGS84 coordinate system, as it uses latitude and longitude values to project the curved surface of the Earth onto a 2D plane. In this projection, only the axes of the plot are transformed, while the actual values of the points remain unchanged.

In [13]:
usaStates.plot {
    geoPolygon()
    coordinatesTransformation = CoordinatesTransformation.mercator()
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="SY4E4g"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-133.102702,57.007526],[-132.932917,56.82131],[-132.620732,56.667956],[-132.653593,56.55294],[-132.817901,56.492694],[-133.042456,56.520078],[-133.201287,56.448878],[-133.420365,56.492694],[-133.66135,56.448878],[-133.710643,56.684386],[-133.688735,56.837741],[-133.869474,56.843218],[-133.907813,56.930849]]],[[[-134.115936,56.48174],[-134.25286,56.558417],[-134.400737,56.722725],[-134.417168,56.848695],[-134.296675,56.908941],[-134.170706,56.848695],[-134.143321,56.952757],[-133.748981,56.772017],[-133.710643,56.596755],[-133.847566,56.574848],[-133.935197,56.377678],[-133.836612,56.322908],[-133.957105,56.092877],[-134.110459,56.142169],[-134.132367,55.999769],[-134.230952,

#### geoMap

`geoMap()` is a basically `geoPolygon()` but it also applies coordinates transformation based on provided `CoordinateReferenceSystem` (`GeoDataFrame.crs`). Now only _WGS84_ is supported (where the mercator projection is applied by default).

In [14]:
// You can see that this plot is identical
// with the previous one.
usaStates.plot {
    geoMap()
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="RsxPko"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-133.102702,57.007526],[-132.932917,56.82131],[-132.620732,56.667956],[-132.653593,56.55294],[-132.817901,56.492694],[-133.042456,56.520078],[-133.201287,56.448878],[-133.420365,56.492694],[-133.66135,56.448878],[-133.710643,56.684386],[-133.688735,56.837741],[-133.869474,56.843218],[-133.907813,56.930849]]],[[[-134.115936,56.48174],[-134.25286,56.558417],[-134.400737,56.722725],[-134.417168,56.848695],[-134.296675,56.908941],[-134.170706,56.848695],[-134.143321,56.952757],[-133.748981,56.772017],[-133.710643,56.596755],[-133.847566,56.574848],[-133.935197,56.377678],[-133.836612,56.322908],[-133.957105,56.092877],[-134.110459,56.142169],[-134.132367,55.999769],[-134.230952,

When the Mercator projection is applied, we can still set axis limits as usual. However, there are inherent boundaries at 180 and 90 degrees due to the nature of geographic coordinates.

In [15]:
usaStates.plot {
    geoMap()
    x.axis.limits = -127..-65
    y.axis.limits = 23..50
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="4Dqqq6"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"xlim":[-127.0,-65.0],
"flip":false,
"ylim":[23.0,50.0]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-133.102702,57.007526],[-132.932917,56.82131],[-132.620732,56.667956],[-132.653593,56.55294],[-132.817901,56.492694],[-133.042456,56.520078],[-133.201287,56.448878],[-133.420365,56.492694],[-133.66135,56.448878],[-133.710643,56.684386],[-133.688735,56.837741],[-133.869474,56.843218],[-133.907813,56.930849]]],[[[-134.115936,56.48174],[-134.25286,56.558417],[-134.400737,56.722725],[-134.417168,56.848695],[-134.296675,56.908941],[-134.170706,56.848695],[-134.143321,56.952757],[-133.748981,56.772017],[-133.710643,56.596755],[-133.847566,56.574848],[-133.

### geoPoints

The `geoPoints()` adds a layer of points constructed using `Point` and `MultiPoint` geometries.

Let's add `worldCities` points over `usaStates` polygons:

In [16]:
usaStates.plot {
    // `geoMap` takes polygons from `geometry`
    // column of `usaStates` inner DataFrame
    geoMap()
    // Add a new dataset using the `worldCities` GeoDataFrame.
    // Layers created within this scope will use it as their base dataset
    // instead of the initial one.
    withData(worldCities) {
        // `geoPoints` takes points from `geometry`
        // column of `worldCities` inner DataFrame
        geoPoints() {
            size = 1.5
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="TuxtGY"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-133.102702,57.007526],[-132.932917,56.82131],[-132.620732,56.667956],[-132.653593,56.55294],[-132.817901,56.492694],[-133.042456,56.520078],[-133.201287,56.448878],[-133.420365,56.492694],[-133.66135,56.448878],[-133.710643,56.684386],[-133.688735,56.837741],[-133.869474,56.843218],[-133.907813,56.930849]]],[[[-134.115936,56.48174],[-134.25286,56.558417],[-134.400737,56.722725],[-134.417168,56.848695],[-134.296675,56.908941],[-134.170706,56.848695],[-134.143321,56.952757],[-133.748981,56.772017],[-133.710643,56.596755],[-133.847566,56.574848],[-133.935197,56.377678],[-133.836612,56.322908],[

## GeoDataFrame modifying

Before plotting, it is often necessary to modify the geo- dataframe. For example, you might filter points within a specific area, translate or scale certain geometries, and so on. `GeoDataFrame` allows direct updates to its inner `DataFrame` using the familiar [DataFrame Operations API](https://kotlin.github.io/dataframe/operations.html).


### DataFrame operations

The function `GeoDataFrame<T>.modify(block: DataFrame<T>.() -> DataFrame<T>): GeoDataFrame<T>` opens a new scope where the receiver is the inner `DataFrame` of this GeoDataFrame. This allows you to perform operations such as `filter`, `take`, `sort`, `update`, and others directly on it. The function returns a GeoDataFrame with the modified DataFrame resulting from the block, while keeping the CRS unchanged.

Let's filter the points in `worldCities`, keeping only those located within the US. To do this, we will first combine all polygons from `usaStates` into a single polygon for convenience:

In [17]:
import org.jetbrains.kotlinx.kandy.letsplot.geo.util.mergePolygons

// experimental function, that merges collection of polygons and
// multipolygons into a single multipolygon
val usaPolygon: MultiPolygon = usaStates.df.geometry.mergePolygons()

In [18]:
plot {
    // `geoPolygon` and `geoMap` can accept a single `Polygon` / `MultiPolygon`
    geoMap(usaPolygon)
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="wnGVqq"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-66.448338,17.984326],[-66.771478,18.006234],[-66.924832,17.929556],[-66.985078,17.973372],[-67.209633,17.956941],[-67.154863,18.19245],[-67.269879,18.362235],[-67.094617,18.515589],[-66.957694,18.488204],[-66.409999,18.488204],[-65.840398,18.433435],[-65.632274,18.367712],[-65.626797,18.203403],[-65.730859,18.186973],[-65.834921,18.017187],[-66.234737,17.929556],[-66.448338,17.984326]]],[[[-75.507197,39.683964],[-75.611259,39.61824],[-75.589352,39.459409],[-75.441474,39.311532],[-75.403136,39.065069],[-75.189535,38.807653],[-75.09095,38.796699],[-75.047134,38.451652],[-75.244304,38.029928],[-75.375751,37.860142],[-75.512674,37.799896],[-75.594828,37.569865],[-75.802952,37.197433],[-75.972737,37.120755],[-76.027507,37.257679],[-75.939876,37.564388],[-75.671506,37.95325],[-75.885106,37.909435],[-75.879629,38.073743],[-75.961783,38.139466],[-75.846768,38.210667],[-76.000122,38.374975],[-76.049415,38.303775],[-76.257538,38.320205],[-76.328738,38.500944],[-76.263015,38.500944],[-76.257538,38.736453],[-76.191815,38.829561],[-76.279446,39.147223],[-76.169907,39.333439],[-76.000122,39.366301],[-75.972737,39.557994],[-76.098707,39.536086],[-76.104184,39.437501],[-76.367077,39.311532],[-76.443754,39.196516],[-76.460185,38.906238],[-76.55877,38.769315],[-76.514954,38.539283],[-76.383508,38.380452],[-76.399939,38.259959],[-76.317785,38.139466],[-76.3616,38.057312],[-76.591632,38.216144],[-76.920248,38.292821],[-77.018833,38.446175],[-77.205049,38.358544],[-77.276249,38.479037],[-77.128372,38.632391],[-77.248864,38.588575],[-77.325542,38.446175],[-77.281726,38.342113],[-77.013356,38.374975],[-76.964064,38.216144],[-76.613539,38.15042],[-76.514954,38.024451],[-76.235631,37.887527],[-76.3616,37.608203],[-76.246584,37.389126],[-76.383508,37.285064],[-76.399939,37.159094],[-76.273969,37.082417],[-76.410893,36.961924],[-76.619016,37.120755],[-76.668309,37.065986],[-76.48757,36.95097],[-75.994645,36.923586],[-75.868676,36.551154],[-75.75366,36.151337],[-76.032984,36.189676],[-76.071322,36.140383],[-76.410893,36.080137],[-76.460185,36.025367],[-76.68474,36.008937],[-76.673786,35.937736],[-76.399939,35.987029],[-76.3616,35.943213],[-76.060368,35.992506],[-75.961783,35.899398],[-75.781044,35.937736],[-75.715321,35.696751],[-75.775568,35.581735],[-75.89606,35.570781],[-76.147999,35.324319],[-76.482093,35.313365],[-76.536862,35.14358],[-76.394462,34.973795],[-76.279446,34.940933],[-76.493047,34.661609],[-76.673786,34.694471],[-76.991448,34.667086],[-77.210526,34.60684],[-77.555573,34.415147],[-77.82942,34.163208],[-77.971821,33.845545],[-78.179944,33.916745],[-78.541422,33.851022],[-78.716684,33.80173],[-78.935762,33.637421],[-79.149363,33.380005],[-79.187701,33.171881],[-79.357487,33.007573],[-79.582041,33.007573],[-79.631334,32.887081],[-79.866842,32.755634],[-79.998289,32.613234],[-80.206412,32.552987],[-80.430967,32.399633],[-80.452875,32.328433],[-80.660998,32.246279],[-80.885553,32.032678],[-81.132015,31.693108],[-81.175831,31.517845],[-81.279893,31.364491],[-81.290846,31.20566],[-81.400385,31.13446],[-81.444201,30.707258],[-81.383954,30.27458],[-81.257985,29.787132],[-80.967707,29.14633],[-80.524075,28.46171

Now, let's create a GeoDataFrame `usaCities` containing only the cities located within the United States. To avoid over plotting, we will select the 30 most populous cities. For this, we will modify `worldCities`:

In [19]:
val usaCities = worldCities.modify {
    // filter the DataFrame to include only points inside the `usaPolygon`
    filter {
        // `usaPolygon.contains(geometry)` checks if the `geometry` (a Point)
        // from the current row of `worldCities` is within the `usaPolygon`
        usaPolygon.contains(geometry)
    }
        // take 30 most populous cities -
        // sort the remaining rows by population size in descending order
        .sortByDesc {
            pop_min
        }
        // and select the top 30 rows
        .take(30)
}


Now we can visualize the result by overlaying the points representing these cities on the polygons of the states (as above):

In [20]:
usaStates.plot {
    geoMap()
    withData(usaCities) {
        geoPoints() {
            tooltips(title = value(name)) {
                line("population", value(pop_min))
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="7aHVPD"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-131.602021,55.117982],[-131.569159,55.28229],[-131.355558,55.183705],[-131.38842,55.01392],[-131.645836,55.035827],[-131.602021,55.117982]]],[[[-131.832052,55.42469],[-131.645836,55.304197],[-131.749898,55.128935],[-131.832052,55.189182],[-131.832052,55.42469]]],[[[-132.976733,56.437924],[-132.735747,56.459832],[-132.631685,56.421493],[-132.664547,56.273616],[-132.878148,56.240754],[-133.069841,56.333862],[-132.976733,56.437924]]],[[[-133.595627,56.350293],[-133.162949,56.317431],[-133.05341,56.125739],[-132.620732,55.912138],[-132.472854,55.780691],[-132.4619,55.671152],[-132.357838,55.649245],[-132.341408,55.506844],[-132.166146,55.364444],[-132.144238,55.238474],[-132.029222,55.276813],[-131.97993,55.178228],[-131.958022,54.789365],[-132.029222,54.701734],[-132.308546,54.718165],[-132.385223,54.915335],[-132.483808,54.898904],[-132.686455,55.046781],[-132.746701,54.997489],[-132.916486,55.046781],[-132.889102,54.898904],[-132.73027,54.937242],[-132.626209,54.882473],[-132.675501,54.679826],[-132.867194,54.701734],[-133.157472,54.95915],[-133.239626,55.090597],[-133.223195,55.22752],[-133.453227,55.216566],[-133.453227,55.320628],[-133.277964,55.331582],[-133.102702,55.42469],[-133.17938,55.588998],[-133.387503,55.62186],[-133.420365,55.884753],[-133.497042,56.0162],[-133.639442,55.923092],[-133.694212,56.070969],[-133.546335,56.142169],[-133.666827,56.311955],[-133.595627,56.350293]]],[[[-133.738027,55.556137],[-133.546335,55.490413],[-133.414888,55.572568],[-133.283441,55.534229],[-133.420365,55.386352],[-133.633966,55.430167],[-133.738027,55.556137]]],[[[-133.907813,56.930849],[-134.050213,57.029434],[-133.885905,57.095157],[-133.343688,57.002049],[-133.102702,57.007526],[-132.932917,56.82131],[-132.620732,56.667956],[-132.653593,56.55294],[-132.817901,56.492694],[-133.042456,56.520078],[-133.201287,56.448878],[-133.420365,56.492694],[-133.66135,56.448878],[-133.710643,56.684386],[-133.688735,56.837741],[-133.869474,56.843218],[-133.907813,56.930849]]],[[[-134.115936,56.48174],[-134.25286,56.558417],[-134.400737,56.722725],[-134.417168,56.848695],[-134.296675,56.908941],[-134.170706,56.848695],[-134.143321,56.952757],[-133.748981,56.772017],[-133.710643,56.596755],[-133.847566,56.574848],[-133.935197,56.377678],[-133.836612,56.322908],[

As you can see, the map of the US is significantly stretched by distant territories such as Puerto Rico, Hawaii, and Alaska. We can remove these regions, keeping only the continental part (48 states):


In [21]:
val usa48 = usaStates.modify {
    filter {
        name !in listOf("Alaska", "Hawaii", "Puerto Rico")
    }
}

usa48.plot { geoMap() }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="25e3Bv"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-109.042503,37.000263],[-109.04798,31.331629],[-111.074448,31.331629],[-112.246513,31.704061],[-114.815198,32.492741],[-114.72209,32.717295],[-114.524921,32.755634],[-114.470151,32.843265],[-114.524921,33.029481],[-114.661844,33.034958],[-114.727567,33.40739],[-114.524921,33.54979],[-114.497536,33.697668],[-114.535874,33.933176],[-114.415382,34.108438],[-114.256551,34.174162],[-114.136058,34.305608],[-114.333228,34.448009],[-114.470151,34.710902],[-114.634459,34.87521],[-114.634459,35.00118],[-114.574213,35.138103],[-114.596121,35.324319],[-114.678275,35.516012],[-114.738521,36.102045],[-114.371566,36.140383],[-114.251074,36.01989],[-114.152489,36.025367],[-114.048427,36.195153],[-114.048427,37.000263],[-110.499369,37.00574],[-109.042503,37.000263]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-94.473842,36.501861],[-90.152536,36.496384],[-90.064905,36.304691],[-90.218259,36.184199],[-90.377091,35.997983],[-89.730812,35.997983],[-89.763673,35.811767],[-89.911551,35.756997],[-89.944412,35.603643],[-90.130628,35.439335],[-90.114197,35.198349],[-90.212782,35.023087],[-90.311367,34.995703],[-90.251121,34.908072],[-90.409952,34.831394],[-90.481152,34.661609],[-90.585214,34.617794],[-90.568783,34.420624],[-90.749522,34.365854],[-90.744046,34.300131],[-90.952169,34.135823],[-90.891923,34.026284],[-91.072662,33.867453],[-91.231493,33.560744],[-91.056231,33.429298],[-91.143862,33.347144],[-91.089093,33.13902],[-91.16577,33.002096],[-93.608485,33.018527],[-94.041164,33.018527],[-94.041164,33.54979],[-94.183564,33.593606],[-94.380734,33.544313],[-94.484796,33.637421],[-94.430026,35.395519],[-94.616242,36.501861],[-94.473842,36.501861]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-123.233256,42.006186],[-122.378853,42.011663],[-121.037003,41.995232],[-120.001861,41.995232],[-119.996384,40.264519],[-120.001861,38.999346],[-118.71478,38.101128],[-117.498899,37.21934],[-116.540435,36.501861],[-115.85034,35.970598],[-114.634459,35.00118],[-114.634459,34.87521],[-114.470151,34.710902],[-114.333228,34.448009],[-114.136058,34.305608],[-114.256551,34.174162],[-114.415382,34.108438],[-114.535874,33.933176],[-114.497536,33.697668],[-114.524921,33.54979],[-114.727567,33.40739],[-114.661844,33.034958],[-114.524921,33.029481],[-114.470151,32.843265],[-114.524921,32.755634],[-114.72209,32.717295],[-116.04751,32.624187],[-117.126467,32.536556],[-117.24696,

### Geometry operations

Another, more elegant way to improve the appearance of the US map is to scale and reposition these polygons, making the plot more compact.

The DataFrame-Geo library provides Kotlin-style extensions for JTS geometries. For instance, `Geometry.translate(x, y)` shifts a geometry by a specified vector, while `Geometry.scaleAroundCenter(factor)` scales a geometry relative to its centroid.


In [22]:
val usaAdjusted = usaStates.modify {
    // custom extensions for `Geometry` based on JTS API;
    // scale and move Alaska
    update { geometry }.where { name == "Alaska" }.with {
        it.scaleAroundCenter(0.5).translate(40.0, -40.0)
    }
        // move Hawaii and Puerto Rico
        .update { geometry }.where { name == "Hawaii" }.with { it.translate(65.0, 0.0) }
        .update { geometry }.where { name == "Puerto Rico" }.with { it.translate(-10.0, 5.0) }
}

usaAdjusted.plot { geoMap() }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="pO1PlK"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-102.0606366614,19.6728748791],[-102.0442056614,19.7550288791],[-101.9374051614,19.7057363791],[-101.9538361614,19.6208438791],[-102.0825441614,19.6317973791],[-102.0606366614,19.6728748791]]],[[[-102.1756521614,19.8262288791],[-102.0825441614,19.7659823791],[-102.1345751614,19.6783513791],[-102.1756521614,19.7084748791],[-102.1756521614,19.8262288791]]],[[[-102.7479926614,20.3328458791],[-102.6274996614,20.3437998791],[-102.5754686614,20.3246303791],[-102.5918996614,20.2506918791],[-102.6987001614,20.2342608791],[-102.7945466614,20.2808148791],[-102.7479926614,20.3328458791]]],[[[-103.0574396614,20.2890303791],[-102.8411006614,20.2725993791],[-102.7863311614,20.1767533791],[-102.5699921614,20.0699528791],[-102.4960531614,20.0042293791],[-102.4905761614,19.9494598791],[-102.4385451614,19.9385063791],[-102.4303301614,19.8673058791],[-102.3426991614,19.7961058791],[-102.3317451614,19.7331208791],[-102.2742371614,19.7522903791],[-102.2495911614,19.7029978791],[-102.2386371614,19.5085663791],[-102.2742371614,19.4647508791],[-102.4138991614,19.4729663791],[-102.4522376614,19.5715513791],[-102.5015301614,19.5633358791],[-102.6028536614,19.6372743791],[-102.6329766614,19.6126283791],[-102.7178691614,19.6372743791],[-102.7041771614,19.5633358791],[-102.6247611614,19.5825048791],[-102.5727306614,19.5551203791],[-102.5973766614,19.4537968791],[-102.6932231614,19.4647508791],[-102.8383621614,19.5934588791],[-102.8794391614,19.6591823791],[-102.8712236614,19.7276438791],[-102.9862396614,19.7221668791],[-102.9862396614,19.7741978791],[-102.8986081614,19.7796748791],[-102.8109771614,19.8262288791],[-102.8493161614,19.9083828791],[-102.9533776614,19.9248138791],[-102.9698086614,20.0562603791],[-103.0081471614,20.1219838791],[-103.0793471614,20.0754298791],[-103.1067321614,20.1493683791],[-103.0327936614,20.1849683791],[-103.0930396614,20.2698613791],[-103.0574396614,20.2890303791]]],[[[-103.1286396614,19.8919523791],[-103.0327936614,19.8590903791],[-102.9670701614,19.9001678791],[-102.9013466614,19.8809983791],[-102.9698086614,19.8070598791],[-103.0766091614,19.8289673791],[-103.1286396614,19.8919523791]]],[[[-103.2135326614,20.5793083791],[-103.2847326614,20.6286008791],[-103.2025786614,20.6614623791],[-102.9314701614,20.6149083791],[-102.8109771614,20.6176468791],[-102.7260846614,20.5245388791],[-102.5699921614,20.4478618791],[-102.58642

An example of the states maps with their centroids:

In [23]:
usa48.plot {
    geoMap()
    withData(usa48.modify { update { geometry }.with { it.centroid } }) {
        geoPoints()
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="MoPYA8"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-109.042503,37.000263],[-109.04798,31.331629],[-111.074448,31.331629],[-112.246513,31.704061],[-114.815198,32.492741],[-114.72209,32.717295],[-114.524921,32.755634],[-114.470151,32.843265],[-114.524921,33.029481],[-114.661844,33.034958],[-114.727567,33.40739],[-114.524921,33.54979],[-114.497536,33.697668],[-114.535874,33.933176],[-114.415382,34.108438],[-114.256551,34.174162],[-114.136058,34.305608],[-114.333228,34.448009],[-114.470151,34.710902],[-114.634459,34.87521],[-114.634459,35.00118],[-114.574213,35.138103],[-114.596121,35.324319],[-114.678275,35.516012],[-114.738521,36.102045],[-114.371566,36.140383],[-114.251074,36.01989],[-114.152489,36.025367],[-114.048427,36.195153],[-114.048427,37.000263],[-110.499369,37.00574],[-109.042503,37.000263]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-94.473842,36.501861],[-90.152536,36.496384],[-90.064905,36.304691],[-90.218259,36.184199],[-90.377091,35.997983],[-89.730812,35.997983],[-89.763673,35.811767],[-89.911551,35.756997],[-89.944412,35.603643],[-90.130628,35.439335],[-90.114197,35.198349],[-90.212782,35.023087],[-90.311367,34.995703],[-90.251121,34.908072],[-90.409952,34.831394],[-90.481152,34.661609],[-90.585214,34.617794],[-90.568783,34.420624],[-90.749522,34.365854],[-90.744046,34.300131],[-90.952169,34.135823],[-90.891923,34.026284],[-91.072662,33.867453],[-91.231493,33.560744],[-91.056231,33.429298],[-91.143862,33.347144],[-91.089093,33.13902],[-91.16577,33.002096],[-93.608485,33.018527],[-94.041164,33.018527],[-94.041164,33.54979],[-94.183564,33.593606],[-94.380734,33.544313],[-94.484796,33.637421],[-94.430026,35.395519],[-94.616242,36.501861],[-94.473842,36.501861]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-123.233256,42.006186],[-122.378853,42.011663],[-121.037003,41.995232],[-120.001861,41.995232],[-119.996384,40.264519],[-120.001861,38.999346],[-118.71478,38.101128],[-117.498899,37.21934],[-116.540435,36.501861],[-115.85034,35.970598],[-114.634459,35.00118],[-114.634459,34.87521],[-114.470151,34.710902],[-114.333228,34.448009],[-114.136058,34.305608],[-114.256551,34.174162],[-114.415382,34.108438],[-114.535874,33.933176],[-114.497536,33.697668],[-114.524921,33.54979],[-114.727567,33.40739],[-114.661844,33.034958],[-114.524921,33.029481],[-114.470151,32.843265],[-114.524921,32.755634

#### Datasets Join

In geo-plotting, separate datasets are often used—one containing the geometries and others with specific data. To combine them, you can [join](https://kotlin.github.io/dataframe/join.html) them using `modify`. Let's load a `DataFrame` with the results of the 2024 US presidential election:


In [24]:
val usa2024electionResults = DataFrame.readCsv("https://gist.githubusercontent.com/AndreiKingsley/348687222aecc4f0eb39e3d81acd515b/raw/a9914352dbdfb426f9146dda633ee382d936b000/usa_2024_election_states.csv")

usa2024electionResults

name,winner
Alabama,Republican
Alaska,Republican
Arizona,Republican
Arkansas,Republican
California,Democrat
Colorado,Democrat
Connecticut,Democrat
Delaware,Democrat
Florida,Republican
Georgia,Republican


And join it to the US states `GeoDataFrame`:

In [25]:
val usaStatesWithElectionResults = usaAdjusted.modify {
    innerJoin(usa2024electionResults) { name }
}

usaStatesWithElectionResults.df

name,geometry,winner
Alabama,"POLYGON ((-87.359296 35.00118, -85.60...",Republican
Alaska,MULTIPOLYGON (((-102.06063666144468 1...,Republican
Arizona,"POLYGON ((-109.042503 37.000263, -109...",Republican
Arkansas,"POLYGON ((-94.473842 36.501861, -90.1...",Republican
California,"POLYGON ((-123.233256 42.006186, -122...",Democrat
Colorado,"POLYGON ((-107.919731 41.003906, -105...",Democrat
Connecticut,"POLYGON ((-73.053528 42.039048, -71.7...",Democrat
Delaware,"POLYGON ((-75.414089 39.804456, -75.5...",Democrat
Florida,"POLYGON ((-85.497137 30.997536, -85.0...",Republican
Georgia,"POLYGON ((-83.109191 35.00118, -83.32...",Republican


Now we can create a geo plot with a color scale based on state election results:

In [26]:
usaStatesWithElectionResults.plot {
    geoMap {
        fillColor(winner) {
            scale = categorical(
                "Republican" to Color.hex("#CC3333"),
                "Democrat" to Color.hex("#3366CC")
            )
        }
        tooltips(name, winner)
    }
    layout {
        title = "USA 2024 President Election Results"
        size = 700 to 500
        style(Style.Void) {
            legend.position = LegendPosition.Top
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="GHg1s3"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"ggtitle":{
"text":"USA 2024 President Election Results"
},
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"ggsize":{
"width":700.0,
"height":500.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"values":["#CC3333","#3366CC"],
"limits":["Republican","Democrat"]
}],
"layers":[{
"mapping":{
"fill":"winner"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"winner":["Republican","Republican","Republican","Republican","Democrat","Democrat","Democrat","Democrat","Republican","Republican","Democrat","Republican","Democrat","Republican","Republican","Republican","Republican","Republican","Democrat","Democrat","Democrat","Republican","Democrat","Republican","Republican","Republican","Republican","Republican","Democrat","Democrat","Democrat","Democrat","Republican","Republican","Republican","Republican","Democrat","Republican","Democrat","Republican","Republican","Republican","Republican","Republican","Democrat","Democrat","Democrat","Republican","Republican","Republican"],
"name":["Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut","Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa","Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan","Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire","New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio","Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota","Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia","Wisconsin","Wyoming"],
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-102.0606366614,19.6728748791],[-102.0442056614,19.7550288791],[-101.9374051614,19.7057363791],[-101.9538361614,19.6208438791],[-102.0825441614,19.6317973791],[-102.0606366614,19.6728748791]]],[[[-102.1756521614,19.8262288791],[-102.0825441614,19.7659823791],[-102.1345751614,19.6783513791],[-102.1756521614,19.7084748791],[-102.1756521614,19.8262288791]]],[[[-102.7479926614,20.3328458791],[-102.6274996614,20.3437998791],[-102.5754686614,20.3246303791],[-102.5918996614,20.2506918791],[-102.6987001614,20.2342608791],[-102.7945466614,20.2808148791],[-102.7479926614,20.3328458791]]],[[[-103.0574396614,20.2890303791],[-102.8411006614,20.2725993791],[-102.7863311614,20.1767533791],[-102.5699921614,20.0699528791],[-102.4960531614,20.0042293791],[-102.4905761614,19.9494598791],[-102.4385451614,19.9385063791],[-102.4303301614,19.8673058791],[-102.3426991614,19.7961058791],[-102.3317451614,19.7331208791],[-102.2742371614,19.7522903791],[-102.2495

### Applying New CRS

A new coordinate system can be applied to a GeoDataFrame by projecting all geometries into it (note that this is not always possible, so proceed with caution).

The *CONUS (Conterminous United States) Albers projection* is a widely used coordinate reference system tailored for the contiguous United States (48 states). It is an equal-area projection, meaning it preserves area proportions while slightly distorting shapes and distances. This projection is ideal for visualizing geographic features across large regions of the continental US.

Let's apply the CONUS Albers projection to the state polygons:



In [27]:
val conusAlbersCrs = CRS.decode("EPSG:5070", true)
val usaAlbers = usa48.applyCrs(conusAlbersCrs)
usaAlbers.crs

PROJCS["NAD83 / Conus Albers", 
  GEOGCS["NAD83", 
    DATUM["North American Datum 1983", 
      SPHEROID["GRS 1980", 6378137.0, 298.257222101, AUTHORITY["EPSG","7019"]], 
      TOWGS84[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 
      AUTHORITY["EPSG","6269"]], 
    PRIMEM["Greenwich", 0.0, AUTHORITY["EPSG","8901"]], 
    UNIT["degree", 0.017453292519943295], 
    AXIS["Geodetic longitude", EAST], 
    AXIS["Geodetic latitude", NORTH], 
    AUTHORITY["EPSG","4269"]], 
  PROJECTION["Albers_Conic_Equal_Area", AUTHORITY["EPSG","9822"]], 
  PARAMETER["central_meridian", -96.0], 
  PARAMETER["latitude_of_origin", 23.0], 
  PARAMETER["standard_parallel_1", 29.5], 
  PARAMETER["false_easting", 0.0], 
  PARAMETER["false_northing", 0.0], 
  PARAMETER["standard_parallel_2", 45.5], 
  UNIT["m", 1.0], 
  AXIS["Easting", EAST], 
  AXIS["Northing", NORTH], 
  AUTHORITY["EPSG","5070"]]

In [28]:
usaAlbers.plot {
    // polygons will work exactly the same -
    // no special coorinates transformation is applied
    // for GeoDF with unsupported crs
    geoMap()
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="XxbodB"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[781042.8727089112,1362464.685305695],[939086.3233899045,1376497.3703536305],[965529.347147124,1282676.0887256158],[1003994.6682262374,1144827.9195802724],[1018200.9453952084,1115095.8197441685],[1030412.2373755082,1098682.8229373556],[1027609.845164306,1087290.9577949676],[1039093.300989175,1081870.9430294372],[1024898.4552310376,1066081.4412098369],[1027022.5552748248,1052196.3663354588],[1020993.2618039548,1032461.8041306012],[1034080.7774663782,1000204.5559356012],[1030735.3444158272,970364.943775331],[1044577.4399077148,941300.2710699852],[998013.0582235336,935397.737183758],[798752.1896982706,915515.1483675538],[796920.5745708892,900680.3058332184],[820174.8928822358,881367.8088375748],[818165.8215500786,862854.467692368],[826308.0549047178,854429.0411888782],[813650.3481693908,836796.685711841],[800869.6313248436,831999.0097170043],[775198.2686367448,848088.5493614418],[770260.2979002206,875149.128652842],[762699.5108830486,877563.2237138974],[755548.2960511462,856200.415926949],[754070.0304500528,835933.4956950658],[728999.4855610648,839380.9295681494],[708213.5366619122,1008236.0916715466],[712511.5609050412,1221311.4729868616],[715387.9176163734,1344434.444400773],[705033.2439916328,1355259.4379862533],[781042.8727089112,1362464.685305695]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-1146224.261422914,1629564.1200067387],[-1233398.6429015228,1002163.839228894],[-1423456.4486094508,1030492.0226274044],[-1526038.6431201661,1089578.8298973192],[-1747087.0889522526,1220221.4665815851],[-1733651.7832731064,1243074.6077685056],[-1714798.73385307,1243692.3856363373],[-1707904.7071025125,1252298.2355996957],[-1708879.9123794255,1273670.4882192523],[-1721229.299220063,1276741.1365282424],[-1719072.0151345772,1318706.7494309533],[-1697627.4726205615,1330662.711232118],[-1691954.7806417693,1346380.3353037548],[-1690317.0910519378,1372878.307382698],[-1675695.641780962,1389962.9133032898],[-1660019.2908362963,1394385.5648633607],[-1646409.6636216603,1406715.4389315187],[-1661035.0540184483,1425776.1111017973],[-1667606.3338877028,1457025.3646339017],[-1678670.4580132565,1477948.7820650944],[-1675924.6597771833,1491772.04823552],[-1667600.1985453228,1505740.3590227182],[-1665490.0365175495,1526566.86948317],[-1668563.0592904272,1549052.9133553922],[-1661020.742259123,1614456.853809582],[-1628078.5862111175,1612318.6892615121],[-1620108.2931252504,1597015.2970832577],[-1611347.6571373148,1595941.3071691655],[-1598632.4739554818,1612854.604034381],[-1581606.1138146226,1701425.6897679088],[-1273221.3249115178,1648716.9025565116],[-1146224.261422914,1629564.1200067387]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[135437.3834752069,1496034.5938211316],[518660.048801957,1510295.335952682],[527762.85992841,1489346.9006474647],[514963.8465521676,1475035.8714703897],[502067.0486928634,1453369.1338512741],[559693.6044266472,1456979.4313934022],[558131.449409189,1435979.7660315044],[545307.7771860413,1429001.8444492642],[543461.8656534968,1411679.879457109],[527906.6387564698,1392273.742853189],[531051.9640481148,1365445.747344456],[523361.6693298852,1345324.3803065466],[514640.264073858,1341727.6622121795],[520676.720050917,1332269.4126844115],[506812.49246154

### geoPath

The `geoPath()` adds a layer of a path constructed using `LineString` and `MultiLineString` geometries.

The following function constructs the shortest path on the Earth's surface, known as a [*great-circle line*](https://en.wikipedia.org/wiki/Great_circle). A great-circle line represents the shortest distance between two points on a sphere, following the curvature of the Earth. The path is approximated using a `LineString` with a specified number of points `n` for precision.


In [29]:
import org.locationtech.jts.geom.*
import kotlin.math.*

fun greatCircleLineString(start: Point, end: Point, n: Int = 100): LineString {
    val factory = GeometryFactory()

    val startLat = Math.toRadians(start.y)
    val startLon = Math.toRadians(start.x)
    val endLat = Math.toRadians(end.y)
    val endLon = Math.toRadians(end.x)

    val deltaLon = endLon - startLon
    val cosStartLat = cos(startLat)
    val cosEndLat = cos(endLat)
    val sinStartLat = sin(startLat)
    val sinEndLat = sin(endLat)
    val a = cosStartLat * cosEndLat * cos(deltaLon) + sinStartLat * sinEndLat
    val angularDistance = acos(a)

    if (angularDistance == 0.0) {
        return factory.createLineString(arrayOf(start.coordinate, end.coordinate))
    }

    val coordinates = mutableListOf<Coordinate>()
    for (i in 0..n) {
        val fraction = i.toDouble() / n
        val sinAngularDistance = sin(angularDistance)
        val A = sin((1 - fraction) * angularDistance) / sinAngularDistance
        val B = sin(fraction * angularDistance) / sinAngularDistance

        val x = A * cosStartLat * cos(startLon) + B * cosEndLat * cos(endLon)
        val y = A * cosStartLat * sin(startLon) + B * cosEndLat * sin(endLon)
        val z = A * sinStartLat + B * sinEndLat

        val lat = atan2(z, sqrt(x * x + y * y))
        val lon = atan2(y, x)

        coordinates.add(Coordinate(Math.toDegrees(lon), Math.toDegrees(lat)))
    }

    return factory.createLineString(coordinates.toTypedArray())
}

This convenient function find city in `usaCities` by name and returns its geometry (point):

In [30]:
fun takeCity(name: String) = usaCities.df.filter { it.name == name }.single().geometry

Use it to take points of New York and Los Angeles:

In [31]:
val newYork = takeCity("New York")
val losAngeles = takeCity("Los Angeles")

Count the shortest path between them:

In [32]:
val curveNY_LA = greatCircleLineString(newYork, losAngeles)

Now, let's plot this curve using `geoPath`, overlaying it on top of the state polygons and highlighting the points corresponding to the cities:

In [33]:
usa48.plot {
    geoMap { alpha = 0.5 }
    geoPath(curveNY_LA) { width = 1.5 }
    geoPoints(listOf(newYork, losAngeles)) {
        size = 8.0
        color = Color.RED
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="qpnhAl"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"alpha":0.5,
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-109.042503,37.000263],[-109.04798,31.331629],[-111.074448,31.331629],[-112.246513,31.704061],[-114.815198,32.492741],[-114.72209,32.717295],[-114.524921,32.755634],[-114.470151,32.843265],[-114.524921,33.029481],[-114.661844,33.034958],[-114.727567,33.40739],[-114.524921,33.54979],[-114.497536,33.697668],[-114.535874,33.933176],[-114.415382,34.108438],[-114.256551,34.174162],[-114.136058,34.305608],[-114.333228,34.448009],[-114.470151,34.710902],[-114.634459,34.87521],[-114.634459,35.00118],[-114.574213,35.138103],[-114.596121,35.324319],[-114.678275,35.516012],[-114.738521,36.102045],[-114.371566,36.140383],[-114.251074,36.01989],[-114.152489,36.025367],[-114.048427,36.195153],[-114.048427,37.000263],[-110.499369,37.00574],[-109.042503,37.000263]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-94.473842,36.501861],[-90.152536,36.496384],[-90.064905,36.304691],[-90.218259,36.184199],[-90.377091,35.997983],[-89.730812,35.997983],[-89.763673,35.811767],[-89.911551,35.756997],[-89.944412,35.603643],[-90.130628,35.439335],[-90.114197,35.198349],[-90.212782,35.023087],[-90.311367,34.995703],[-90.251121,34.908072],[-90.409952,34.831394],[-90.481152,34.661609],[-90.585214,34.617794],[-90.568783,34.420624],[-90.749522,34.365854],[-90.744046,34.300131],[-90.952169,34.135823],[-90.891923,34.026284],[-91.072662,33.867453],[-91.231493,33.560744],[-91.056231,33.429298],[-91.143862,33.347144],[-91.089093,33.13902],[-91.16577,33.002096],[-93.608485,33.018527],[-94.041164,33.018527],[-94.041164,33.54979],[-94.183564,33.593606],[-94.380734,33.544313],[-94.484796,33.637421],[-94.430026,35.395519],[-94.616242,36.501861],[-94.473842,36.501861]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-123.233256,42.006186],[-122.378853,42.011663],[-121.037003,41.995232],[-120.001861,41.995232],[-119.996384,40.264519],[-120.001861,38.999346],[-118.71478,38.101128],[-117.498899,37.21934],[-116.540435,36.501861],[-115.85034,35.970598],[-114.634459,35.00118],[-114.634459,34.87521],[-114.470151,34.710902],[-114.333228,34.448009],[-114.136058,34.305608],[-114.256551,34.174162],[-114.415382,34.108438],[-114.535874,33.933176],[-114.497536,33.697668],[-114.524921,33.54979],[-114.727567,33.4073

### geoRectangles

The `geoRectangles()` adds a layer of rectangles constructed using `Envelope`. The `Envelope` class represents a rectangular region in the coordinate space, defined by its minimum and maximum coordinates. It is commonly used for bounding boxes, spatial indexing, and efficient geometric calculations.


Let's get `usa48` common bounding box:

In [34]:
// The `.bounds()` function calculates the minimum bounding box
// of all geometries in the `geometry` column of a `GeoDataFrame`,
// returning it as an `Envelope`.
val usa48Bounds: Envelope = usa48.bounds().also {
    // JTS API for in-place envelope expansion
    it.expandBy(1.0)
}

And plot it with the polygon plot:

In [35]:
usa48.plot {
    geoMap()
    geoRectangles(usa48Bounds) {
        alpha = 0.0
        borderLine {
            width = 2.0
            color = Color.GREY
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="IeLscE"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-109.042503,37.000263],[-109.04798,31.331629],[-111.074448,31.331629],[-112.246513,31.704061],[-114.815198,32.492741],[-114.72209,32.717295],[-114.524921,32.755634],[-114.470151,32.843265],[-114.524921,33.029481],[-114.661844,33.034958],[-114.727567,33.40739],[-114.524921,33.54979],[-114.497536,33.697668],[-114.535874,33.933176],[-114.415382,34.108438],[-114.256551,34.174162],[-114.136058,34.305608],[-114.333228,34.448009],[-114.470151,34.710902],[-114.634459,34.87521],[-114.634459,35.00118],[-114.574213,35.138103],[-114.596121,35.324319],[-114.678275,35.516012],[-114.738521,36.102045],[-114.371566,36.140383],[-114.251074,36.01989],[-114.152489,36.025367],[-114.048427,36.195153],[-114.048427,37.000263],[-110.499369,37.00574],[-109.042503,37.000263]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-94.473842,36.501861],[-90.152536,36.496384],[-90.064905,36.304691],[-90.218259,36.184199],[-90.377091,35.997983],[-89.730812,35.997983],[-89.763673,35.811767],[-89.911551,35.756997],[-89.944412,35.603643],[-90.130628,35.439335],[-90.114197,35.198349],[-90.212782,35.023087],[-90.311367,34.995703],[-90.251121,34.908072],[-90.409952,34.831394],[-90.481152,34.661609],[-90.585214,34.617794],[-90.568783,34.420624],[-90.749522,34.365854],[-90.744046,34.300131],[-90.952169,34.135823],[-90.891923,34.026284],[-91.072662,33.867453],[-91.231493,33.560744],[-91.056231,33.429298],[-91.143862,33.347144],[-91.089093,33.13902],[-91.16577,33.002096],[-93.608485,33.018527],[-94.041164,33.018527],[-94.041164,33.54979],[-94.183564,33.593606],[-94.380734,33.544313],[-94.484796,33.637421],[-94.430026,35.395519],[-94.616242,36.501861],[-94.473842,36.501861]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-123.233256,42.006186],[-122.378853,42.011663],[-121.037003,41.995232],[-120.001861,41.995232],[-119.996384,40.264519],[-120.001861,38.999346],[-118.71478,38.101128],[-117.498899,37.21934],[-116.540435,36.501861],[-115.85034,35.970598],[-114.634459,35.00118],[-114.634459,34.87521],[-114.470151,34.710902],[-114.333228,34.448009],[-114.136058,34.305608],[-114.256551,34.174162],[-114.415382,34.108438],[-114.535874,33.933176],[-114.497536,33.697668],[-114.524921,33.54979],[-114.727567,33.40739],[-114.661844,33.034958],[-114.524921,33.029481],[-114.470151,32.843265],[-114.524921,32.755634],[-114.72209,32.717295],[-116.04751,32.624187],[-117.126467,32.536556],[-117.24696,

In addition, `geoRectangles` also works with polygons and multipolygon. In such cases, the bounding box of each geometry will be calculated and used individually:

In [36]:
usa48.plot {
    geoMap()
    geoRectangles()
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="gs6hBj"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"Polygon\",\"coordinates\":[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-109.042503,37.000263],[-109.04798,31.331629],[-111.074448,31.331629],[-112.246513,31.704061],[-114.815198,32.492741],[-114.72209,32.717295],[-114.524921,32.755634],[-114.470151,32.843265],[-114.524921,33.029481],[-114.661844,33.034958],[-114.727567,33.40739],[-114.524921,33.54979],[-114.497536,33.697668],[-114.535874,33.933176],[-114.415382,34.108438],[-114.256551,34.174162],[-114.136058,34.305608],[-114.333228,34.448009],[-114.470151,34.710902],[-114.634459,34.87521],[-114.634459,35.00118],[-114.574213,35.138103],[-114.596121,35.324319],[-114.678275,35.516012],[-114.738521,36.102045],[-114.371566,36.140383],[-114.251074,36.01989],[-114.152489,36.025367],[-114.048427,36.195153],[-114.048427,37.000263],[-110.499369,37.00574],[-109.042503,37.000263]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-94.473842,36.501861],[-90.152536,36.496384],[-90.064905,36.304691],[-90.218259,36.184199],[-90.377091,35.997983],[-89.730812,35.997983],[-89.763673,35.811767],[-89.911551,35.756997],[-89.944412,35.603643],[-90.130628,35.439335],[-90.114197,35.198349],[-90.212782,35.023087],[-90.311367,34.995703],[-90.251121,34.908072],[-90.409952,34.831394],[-90.481152,34.661609],[-90.585214,34.617794],[-90.568783,34.420624],[-90.749522,34.365854],[-90.744046,34.300131],[-90.952169,34.135823],[-90.891923,34.026284],[-91.072662,33.867453],[-91.231493,33.560744],[-91.056231,33.429298],[-91.143862,33.347144],[-91.089093,33.13902],[-91.16577,33.002096],[-93.608485,33.018527],[-94.041164,33.018527],[-94.041164,33.54979],[-94.183564,33.593606],[-94.380734,33.544313],[-94.484796,33.637421],[-94.430026,35.395519],[-94.616242,36.501861],[-94.473842,36.501861]]]}","{\"type\":\"Polygon\",\"coordinates\":[[[-123.233256,42.006186],[-122.378853,42.011663],[-121.037003,41.995232],[-120.001861,41.995232],[-119.996384,40.264519],[-120.001861,38.999346],[-118.71478,38.101128],[-117.498899,37.21934],[-116.540435,36.501861],[-115.85034,35.970598],[-114.634459,35.00118],[-114.634459,34.87521],[-114.470151,34.710902],[-114.333228,34.448009],[-114.136058,34.305608],[-114.256551,34.174162],[-114.415382,34.108438],[-114.535874,33.933176],[-114.497536,33.697668],[-114.524921,33.54979],[-114.727567,33.40739],[-114.661844,33.034958],[-114.524921,33.029481],[-114.470151,32.843265],[-114.524921,32.755634],[-114.72209,32.717295],[-116.04751,32.624187],[-117.126467,32.536556],[-117.24696,

## Write GeoDataFrame

A `GeoDataFrame` can be saved to a file in both GeoJSON and Shapefile formats using the `GeoDataFrame.write..(filename)` functions.

### GeoJSON

Let's save the modified GeoDataFrame containing US cities, which was initially in Shapefile format, to a GeoJSON file.


In [37]:
usaCities.writeGeoJson("usa_cities.geojson")

In [38]:
GeoDataFrame.readGeoJson("usa_cities.geojson").plot { geoPoints() }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="0Pagtt"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"point",
"map":{
"geometry":["{\"type\":\"Point\",\"coordinates\":[-86.7819,36.1719]}","{\"type\":\"Point\",\"coordinates\":[-97.5206,35.472]}","{\"type\":\"Point\",\"coordinates\":[-122.6819,45.522]}","{\"type\":\"Point\",\"coordinates\":[-77.0114,38.9015]}","{\"type\":\"Point\",\"coordinates\":[-106.6413,35.105]}","{\"type\":\"Point\",\"coordinates\":[-106.5119,31.7819]}","{\"type\":\"Point\",\"coordinates\":[-72.9,41.3304]}","{\"type\":\"Point\",\"coordinates\":[-122.3153,47.6004]}","{\"type\":\"Point\",\"coordinates\":[-87.9167,43.0296]}","{\"type\":\"Point\",\"coordinates\":[-71.072,42.3319]}","{\"type\":\"Point\",\"coordinates\":[-76.6146,39.2815]}","{\"type\":\"Point\",\"coordinates\":[-97.7447,30.2689]}","{\"type\":\"Point\",\"coordinates\":[-82.992,39.9819]}","{\"type\":\"Point\",\"coordinates\":[-97.34,32.74]}","{\"type\":\"Point\",\"coordinates\":[-86.172,39.7519]}","{\"type\":\"Point\",\"coordinates\":[-81.6719,30.332]}","{\"type\":\"Point\",\"coordinates\":[-80.832,35.2069]}","{\"type\":\"Point\",\"coordinates\":[-121.8882,37.3309]}","{\"type\":\"Point\",\"coordinates\":[-83.0513,42.3336]}","{\"type\":\"Point\",\"coordinates\":[-117.1509,32.7189]}","{\"type\":\"Point\",\"coordinates\":[-96.7947,32.772]}","{\"type\":\"Point\",\"coordinates\":[-80.0019,40.4319]}","{\"type\":\"Point\",\"coordinates\":[-98.4927,29.4198]}","{\"type\":\"Point\",\"coordinates\":[-112.0674,33.4483]}","{\"type\":\"Point\",\"coordinates\":[-75.1798,39.9459]}","{\"type\":\"Point\",\"coordinates\":[-104.986,39.7411]}","{\"type\":\"Point\",\"coordinates\":[-87.6352,41.848]}","{\"type\":\"Point\",\"coordinates\":[-95.3484,29.7413]}","{\"type\":\"Point\",\"coordinates\":[-118.232,34.0492]}","{\"type\":\"Point\",\"coordinates\":[-73.9957,40.7216]}"]
},
"data":{
}
}],
"spec_id":"50"
};
 var containerDiv = document.getElementById("0Pagtt");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 600.0,
 height: 400.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -120 
 
 
 
 
 
 
 
 
 -110 
 
 
 
 
 
 
 
 
 -100 
 
 
 
 
 
 
 
 
 -90 
 
 
 
 
 
 
 
 
 -80 
 
 
 
 
 
 
 
 
 -70 
 
 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 32 
 
 
 
 
 
 
 34 
 
 
 
 
 
 
 36 
 
 
 
 
 
 
 38 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 42 
 
 
 
 
 
 
 44 
 
 
 
 
 
 
 46 
 
 
 
 
 
 
 48 
 
 
 
 
 
 
 
 
 lat 
 
 
 
 
 lon

### Shapefile

Unlike GeoJSON, Shapefile supports **only one type of geometry**.

Let's save the GeoDataFrame containing the boundaries of US states, which was initially in GeoJSON format and included both polygons and multipolygons, to a Shapefile. To do this, we will first cast all geometries to `MultiPolygon`.


In [39]:
// All geometries should be the same type - Shapefile restriction,
// but we have `Polygon` and `MultiPolygon`.
// Cast them all into multipolygons
usa48.modify {
    convert { geometry }.with {
        when(it) {
            // Casts `Polygon` to a `MultiPolygon` with a single entity
            is Polygon -> it.toMultiPolygon()
            is MultiPolygon -> it
            else -> error("not a polygonal")
        }
    }
}
    // All files comprising the Shapefile will be saved to
    // a directory named "usa_48" and will have the same base name.
    .writeShapefile("usa_48")

In [40]:
GeoDataFrame.readShapefile("usa_48/usa_48.shp").plot { geoMap() }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="On8s7a"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"coord":{
"name":"map",
"flip":false
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"map_data_meta":{
"geodataframe":{
"geometry":"geometry"
}
},
"geom":"polygon",
"map":{
"geometry":["{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-87.359296,35.00118],[-85.606675,34.984749],[-85.431413,34.124869],[-85.184951,32.859696],[-85.069935,32.580372],[-84.960397,32.421541],[-85.004212,32.322956],[-84.889196,32.262709],[-85.058981,32.13674],[-85.053504,32.01077],[-85.141136,31.840985],[-85.042551,31.539753],[-85.113751,31.27686],[-85.004212,31.003013],[-85.497137,30.997536],[-87.600282,30.997536],[-87.633143,30.86609],[-87.408589,30.674397],[-87.446927,30.510088],[-87.37025,30.427934],[-87.518128,30.280057],[-87.655051,30.247195],[-87.90699,30.411504],[-87.934375,30.657966],[-88.011052,30.685351],[-88.10416,30.499135],[-88.137022,30.318396],[-88.394438,30.367688],[-88.471115,31.895754],[-88.241084,33.796253],[-88.098683,34.891641],[-88.202745,34.995703],[-87.359296,35.00118]]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-109.042503,37.000263],[-109.04798,31.331629],[-111.074448,31.331629],[-112.246513,31.704061],[-114.815198,32.492741],[-114.72209,32.717295],[-114.524921,32.755634],[-114.470151,32.843265],[-114.524921,33.029481],[-114.661844,33.034958],[-114.727567,33.40739],[-114.524921,33.54979],[-114.497536,33.697668],[-114.535874,33.933176],[-114.415382,34.108438],[-114.256551,34.174162],[-114.136058,34.305608],[-114.333228,34.448009],[-114.470151,34.710902],[-114.634459,34.87521],[-114.634459,35.00118],[-114.574213,35.138103],[-114.596121,35.324319],[-114.678275,35.516012],[-114.738521,36.102045],[-114.371566,36.140383],[-114.251074,36.01989],[-114.152489,36.025367],[-114.048427,36.195153],[-114.048427,37.000263],[-110.499369,37.00574],[-109.042503,37.000263]]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-94.473842,36.501861],[-90.152536,36.496384],[-90.064905,36.304691],[-90.218259,36.184199],[-90.377091,35.997983],[-89.730812,35.997983],[-89.763673,35.811767],[-89.911551,35.756997],[-89.944412,35.603643],[-90.130628,35.439335],[-90.114197,35.198349],[-90.212782,35.023087],[-90.311367,34.995703],[-90.251121,34.908072],[-90.409952,34.831394],[-90.481152,34.661609],[-90.585214,34.617794],[-90.568783,34.420624],[-90.749522,34.365854],[-90.744046,34.300131],[-90.952169,34.135823],[-90.891923,34.026284],[-91.072662,33.867453],[-91.231493,33.560744],[-91.056231,33.429298],[-91.143862,33.347144],[-91.089093,33.13902],[-91.16577,33.002096],[-93.608485,33.018527],[-94.041164,33.018527],[-94.041164,33.54979],[-94.183564,33.593606],[-94.380734,33.544313],[-94.484796,33.637421],[-94.430026,35.395519],[-94.616242,36.501861],[-94.473842,36.501861]]]]}","{\"type\":\"MultiPolygon\",\"coordinates\":[[[[-123.233256,42.006186],[-122.378853,42.011663],[-121.037003,41.995232],[-120.001861,41.995232],[-119.996384,40.264519],[-120.001861,38.999346],[-118.71478,38.101128],[-117.498899,37.21934],[-116.540435,36.501861],[-115.85034,35.970598],[-114.634459,35.00118],[-114.634459,34.87521],[-114.470151,34.710902],[-114.333228,34.448009],[-114.136058,34.305608],[-114.256551,34.174162],[-114.415382,34.108438],[-114.535874,33.933176],[-114.497536,33.697668],[-114.524921,33.54979],[-114.727567,33.40739],[-114.661844,33.034958],[-114.524921,33.029481],[-114.470151,32.843265],[-114.524921,32.755634],[-114.72209,32.717295],[-116.04751,32.624187],[-117.126